# 02 · Restricted Hartree-Fock (RHF) with DIIS

RHF 是量子化学最基本的方法——它是所有波函数方法（MP2, CCSD, CASSCF……）的起点。

本 notebook 从 Roothaan 方程出发，完整实现带 DIIS 加速的 RHF SCF 程序。


---

## 1 · Roothaan-Hall 方程

### 1.1 核心方程

在基组 $\{\chi_\mu\}$ 下展开分子轨道 $\phi_i = \sum_\mu C_{\mu i} \chi_\mu$，变分极小化得到：

$$
\boxed{\mathbf{F} \mathbf{C} = \mathbf{S} \mathbf{C} \boldsymbol{\varepsilon}}
$$

| 矩阵 | 含义 | 表达式 |
|------|------|--------|
| $\mathbf{S}$ | 重叠矩阵 | $S_{\mu\nu} = \langle \chi_\mu | \chi_\nu \rangle$ |
| $\mathbf{F}$ | Fock 矩阵 | $F_{\mu\nu} = H^{\text{core}}_{\mu\nu} + \sum_{\kappa\lambda} D_{\kappa\lambda} \big[(\mu\nu|\kappa\lambda) - \frac{1}{2}(\mu\kappa|\nu\lambda)\big]$ |
| $\mathbf{C}$ | MO 系数 | 每列是一个分子轨道 |
| $\boldsymbol{\varepsilon}$ | 轨道能量 (对角) | $\varepsilon_i$ 是第 $i$ 个 MO 的能量 |

### 1.2 Fock 矩阵的组成

$$
\mathbf{F} = \mathbf{H}^{\text{core}} + \mathbf{J} - \frac{1}{2}\mathbf{K}
$$

- **Coulomb**: $J_{\mu\nu} = \sum_{\kappa\lambda} D_{\kappa\lambda} (\mu\nu|\kappa\lambda)$
- **Exchange**: $K_{\mu\nu} = \sum_{\kappa\lambda} D_{\kappa\lambda} (\mu\kappa|\nu\lambda)$

### 1.3 密度矩阵

闭壳层 RHF，每个占据轨道 2 个电子：

$$
D_{\mu\nu} = 2 \sum_i^{N/2} C_{\mu i} C_{\nu i}^*
$$

用矩阵写法：$\mathbf{D} = \mathbf{C}_{\text{occ}} \, \text{diag}(2,2,...,2) \, \mathbf{C}_{\text{occ}}^T$

### 1.4 总能量

$$
E_{\text{total}} = \underbrace{\text{Tr}(\mathbf{D} \mathbf{H}^{\text{core}}) + \frac{1}{2}\text{Tr}(\mathbf{D} \mathbf{V}_{\text{eff}})}_{E_{\text{elec}}} + \underbrace{\sum_{A<B} \frac{Z_A Z_B}{R_{AB}}}_{E_{\text{nn}}}
$$

### 1.5 SCF 迭代

Fock 矩阵依赖密度矩阵，密度矩阵又来自 Fock 矩阵——这是非线性自洽问题：

```
初始猜测 D0
  -> 构建 F(D0)
    -> 对角化 F C = S C epsilon
      -> 计算新 D = C_occ n C_occ^T
        -> D 变化 < 阈值?  是 -> 收敛!
        -> 否 -> 用 D_new 回到构建 F
```


---

## 2 · 分子设定与积分


In [30]:
from pyscf import gto
import numpy as np
from scipy.linalg import eigh, fractional_matrix_power
import numpy.typing as npt

# -------- 分子定义: H2O / cc-pVDZ --------
mol = gto.M(atom="O 0 0 0; H 0 -0.757 0.587; H 0 0.757 0.587", basis="cc-pvdz")
nao = mol.nao_nr()
nelec = mol.nelectron

# -------- 积分 --------
S = mol.intor('int1e_ovlp_sph')       # S_{mu nu} = <chi_mu | chi_nu>
T = mol.intor('int1e_kin_sph')        # T_{mu nu} = <chi_mu | -1/2 nabla^2 | chi_nu>
V = mol.intor('int1e_nuc_sph')        # V_{mu nu} = <chi_mu | -sum_A Z_A/r_A | chi_nu>
H_core = T + V                         # 芯 Hamiltonian
A = fractional_matrix_power(S, -0.5)   # 正交化矩阵 S^{-1/2}
eri = mol.intor('int2e')               # 双电子积分 (mu nu | kappa lambda)

# -------- 核排斥能 E_nn = sum_{A<B} Z_A Z_B / R_AB --------
def compute_nuclear_repulsion(mol):
    coords = mol.atom_coords()
    charges = mol.atom_charges()
    natm = len(charges)
    E_nn = 0.0
    for i in range(natm):
        for j in range(i + 1, natm):
            r_ij = np.linalg.norm(coords[i] - coords[j])
            E_nn += charges[i] * charges[j] / r_ij
    return E_nn

E_nn = compute_nuclear_repulsion(mol)

print(f"nao = {nao}, nelec = {nelec}")
print(f"S shape = {S.shape}, H_core shape = {H_core.shape}")
print(f"eri shape = {eri.shape} ({eri.size:,} elements)")
print(f"E_nn = {E_nn:.10f} Hartree")


nao = 24, nelec = 10
S shape = (24, 24), H_core shape = (24, 24)
eri shape = (24, 24, 24, 24) (331,776 elements)
E_nn = 9.1882584177 Hartree


---

## 3 · SCF 核心函数

逐个实现 SCF 循环的每一步。


In [ ]:
# ============================================================
# 3.1 占据数: 按能量排序，最低 N/2 个轨道各占 2 个电子
# ============================================================
def get_occ(mo_energy, mo_coeff):
    e_idx = np.argsort(mo_energy)
    mo_occ = np.zeros_like(mo_energy)
    nocc = nelec // 2
    mo_occ[e_idx[:nocc]] = 2
    return mo_occ


# ============================================================
# 3.2 密度矩阵: D = C_occ * diag(n_occ) * C_occ^T
# ============================================================
def make_rdm1(mo_coeff, mo_occ):
    mocc = mo_coeff[:, mo_occ > 0]         # 占据 MO 系数 (nao x nocc)
    occ = mo_occ[mo_occ > 0]               # 占据数 [2, 2, ..., 2]
    return (mocc * occ).dot(mocc.conj().T)


# ============================================================
# 3.3 J 和 K 矩阵
# J_{mu,nu} = sum_{kl} D_{kl} * (mu nu | k l)        <- einsum "ijkl,kl->ij"
# K_{mu,nu} = sum_{kl} D_{kl} * (mu k | nu l)        <- einsum "ikjl,kl->ij"
# ============================================================
def get_jk(eri, dm):
    vj = np.einsum("ijkl,kl->ij", eri, dm, optimize=True)
    vk = np.einsum("ikjl,kl->ij", eri, dm, optimize=True)
    return vj, vk


# ============================================================
# 3.4 有效势: V_eff = J - 1/2 K
# ============================================================
def get_veff(eri, dm):
    vj, vk = get_jk(eri, dm)
    return vj - 0.5 * vk


# ============================================================
# 3.5 Fock 矩阵: F = H_core + V_eff
# ============================================================
def get_fock(H_core, V_eff):
    return H_core + V_eff


# ============================================================
# 3.6 电子能量: E_elec = Tr(D H_core) + 1/2 Tr(D V_eff)
# ============================================================
def energy_elec(dm, H_core, V_eff):
    e1 = np.einsum("ij,ji->", H_core, dm, optimize=True)
    e2 = np.einsum("ij,ji->", V_eff, dm, optimize=True)
    return e1 + 0.5 * e2, 0.5 * e2


def energy_tot(dm, H_core, V_eff):
    return E_nn + energy_elec(dm, H_core, V_eff)[0]


# ============================================================
# 3.7 初猜密度: 对角化 H_core 得到初始 MO
# ============================================================
def get_init_guess(H_core, S):
    mo_energy, mo_coeff = eigh(H_core, S)
    mo_occ = get_occ(mo_energy, mo_coeff)
    return make_rdm1(mo_coeff, mo_occ)


print("All helper functions ready.")


All helper functions ready.


---

## 4 · 基础 SCF 迭代 (无 DIIS)

先跑一遍最简单的 SCF，理解自洽迭代过程。


In [32]:
# -------- 基础 SCF (无加速) --------
max_cycle = 50
conv_tol = 1e-8
e_old = 0.0
dm = get_init_guess(H_core, S)

print(f"{'Iter':>4s}  {'E_total':>16s}  {'dE':>10s}  {'dD':>10s}")
print("-" * 46)

for cycle in range(max_cycle):
    V_eff = get_veff(eri, dm)                        # 1. J - 1/2 K
    fock = get_fock(H_core, V_eff)                   # 2. F = H_core + V_eff
    e_tot = energy_tot(dm, H_core, V_eff)             # 3. 总能量

    mo_energy, mo_coeff = eigh(fock, S)               # 4. 对角化 F C = S C eps
    mo_occ = get_occ(mo_energy, mo_coeff)             # 5. 占据数
    dm_new = make_rdm1(mo_coeff, mo_occ)              # 6. 新密度 D_new

    e_diff = abs(e_tot - e_old)
    d_norm = np.linalg.norm(dm_new - dm)

    print(f"{cycle+1:4d}  {e_tot:16.10f}  {e_diff:10.3e}  {d_norm:10.3e}")

    if e_diff < conv_tol and d_norm < conv_tol:
        print(f"\nConverged in {cycle+1} iterations")
        print(f"E_total (no DIIS) = {e_tot:.10f} Hartree")
        break

    dm = dm_new
    e_old = e_tot
else:
    print("WARNING: SCF did not converge!")


Iter           E_total          dE          dD
----------------------------------------------
   1    -68.8738900069   6.887e+01   1.531e+01
   2    -69.9486889568   1.075e+00   1.469e+01
   3    -73.3050793488   3.356e+00   7.425e+00
   4    -73.4263029930   1.212e-01   7.145e+00
   5    -74.6942001206   1.268e+00   2.455e+00
   6    -75.5219682443   8.278e-01   1.530e+00
   7    -75.8505484865   3.286e-01   8.534e-01
   8    -75.9662153709   1.157e-01   5.145e-01
   9    -76.0065148306   4.030e-02   2.894e-01
  10    -76.0199860973   1.347e-02   1.703e-01
  11    -76.0245101874   4.524e-03   9.698e-02
  12    -76.0260140658   1.504e-03   5.646e-02
  13    -76.0265156248   5.016e-04   3.237e-02
  14    -76.0266824251   1.668e-04   1.876e-02
  15    -76.0267379710   5.555e-05   1.079e-02
  16    -76.0267564523   1.848e-05   6.237e-03
  17    -76.0267626044   6.152e-06   3.592e-03
  18    -76.0267646517   2.047e-06   2.075e-03
  19    -76.0267653332   6.814e-07   1.196e-03
  20    -76.0

---

## 5 · Pulay 的 DIIS 加速

### 5.1 动机

基础 SCF 对某些体系收敛很慢甚至会振荡发散。DIIS (Direct Inversion in the Iterative Subspace) 是 Pulay (1980) 提出的外推方法——不改变物理，只是聪明地组合历史 Fock 矩阵来加速。

### 5.2 误差向量

定义第 $i$ 次迭代的误差向量：

$$
\mathbf{e}_i = \mathbf{S}^{-1/2} \big( \mathbf{F}_i \mathbf{D}_i \mathbf{S} - \mathbf{S} \mathbf{D}_i \mathbf{F}_i \big) \mathbf{S}^{-1/2}
$$

**物理含义**：SCF 收敛时 $\mathbf{F}$ 和 $\mathbf{D}$ 对易（共享同一组本征矢），此时 $\mathbf{e} = \mathbf{0}$。$\mathbf{e}_i$ 衡量第 $i$ 次迭代离收敛有多远。

### 5.3 DIIS 方程组

假设误差向量在迭代子空间中近似线性相关，寻找系数 $c_i$ 使误差的线性组合最小：

$$
\sum_i c_i \mathbf{e}_i \approx \mathbf{0}, \quad \sum_i c_i = 1
$$

这等价于解：

$$
\begin{pmatrix}
\langle e_1|e_1\rangle & \cdots & \langle e_1|e_n\rangle & -1 \\
\vdots & \ddots & \vdots & \vdots \\
\langle e_n|e_1\rangle & \cdots & \langle e_n|e_n\rangle & -1 \\
-1 & \cdots & -1 & 0
\end{pmatrix}
\begin{pmatrix} c_1 \\ \vdots \\ c_n \\ \lambda \end{pmatrix}
= \begin{pmatrix} 0 \\ \vdots \\ 0 \\ -1 \end{pmatrix}
$$

其中 $\langle e_i|e_j\rangle = \text{Tr}(\mathbf{e}_i^T \mathbf{e}_j)$ 是误差向量的 Frobenius 内积。

最后用同样的系数组合 Fock 矩阵：

$$
\mathbf{F}^{\text{DIIS}} = \sum_{i=1}^n c_i \, \mathbf{F}_i
$$

然后用 $\mathbf{F}^{\text{DIIS}}$ 对角化得到新的 MO 和密度矩阵。


In [33]:
# ============================================================
# 5.4 DIIS 实现
# ============================================================

def get_err_vec(fock, dm, S, A):
    # e = A @ (F D S - S D F) @ A,  其中 A = S^{-1/2}
    # 在正交基下度量 F 和 D 的对易程度
    return A @ (fock @ dm @ S - S @ dm @ fock) @ A


def apply_diis(fock_list, err_vec_list):
    # 1. 构建 B 矩阵: B_ij = <e_i | e_j> = Tr(e_i^T e_j)
    #    最后一行/列是 Lagrange 乘子约束 sum(c_i) = 1
    n = len(fock_list)
    dim = n + 1
    B = np.empty((dim, dim))
    B[-1, :] = -1
    B[:, -1] = -1
    B[-1, -1] = 0

    for i in range(n):
        for j in range(n):
            B[i, j] = np.einsum("ij,ij->", err_vec_list[i], err_vec_list[j], optimize=True)

    rhs = np.zeros(dim)
    rhs[-1] = -1
    coeff = np.linalg.solve(B, rhs)

    # 2. F_DIIS = sum_i c_i F_i
    F_new = np.einsum("i,ikl->kl", coeff[:-1], fock_list)
    return F_new


print("DIIS functions defined.")


DIIS functions defined.


---

## 6 · SCF + DIIS 完整运行

将 DIIS 嵌入 SCF 循环：
- 前 2 次迭代不启动 DIIS（需要积累历史信息）
- 每次迭代记录 F_i 和 e_i
- 从第 3 次开始，用 DIIS 外推 Fock 矩阵


In [34]:
# -------- SCF + DIIS --------
max_cycle = 50
conv_tol = 1e-8
e_old = 0.0
fock_list = []          # 历史 Fock 矩阵
err_vec_list = []       # 历史误差向量

dm = get_init_guess(H_core, S)

print(f"{'Iter':>4s}  {'E_total':>16s}  {'dE':>10s}  {'dD':>10s}")
print("-" * 46)

for cycle in range(max_cycle):
    V_eff = get_veff(eri, dm)                        # 1. 有效势 J - 1/2 K
    fock = get_fock(H_core, V_eff)                   # 2. Fock 矩阵
    e_tot = energy_tot(dm, H_core, V_eff)             # 3. 能量

    err = get_err_vec(fock, dm, S, A)                 # 4. 误差向量
    fock_list.append(fock)
    err_vec_list.append(err)

    # ---- DIIS 外推 (从第 3 次迭代开始) ----
    if cycle >= 2:
        fock = apply_diis(fock_list, err_vec_list)

    mo_energy, mo_coeff = eigh(fock, S)               # 5. 对角化
    mo_occ = get_occ(mo_energy, mo_coeff)             # 6. 占据
    dm_new = make_rdm1(mo_coeff, mo_occ)              # 7. 新密度

    e_diff = abs(e_tot - e_old)
    d_norm = np.linalg.norm(dm_new - dm)

    print(f"{cycle+1:4d}  {e_tot:16.10f}  {e_diff:10.3e}  {d_norm:10.3e}")

    if e_diff < conv_tol and d_norm < conv_tol:
        print(f"\nConverged in {cycle+1} iterations!")
        print(f"E_total (RHF+DIIS/cc-pVDZ) = {e_tot:.10f} Hartree")
        print(f"E_elec = {e_tot - E_nn:.10f}")
        print(f"E_nn   = {E_nn:.10f}")

        # 打印前几个轨道能量
        print(f"\nOrbital energies (Hartree):")
        e_idx = np.argsort(mo_energy)
        for i in range(min(nelec//2 + 5, len(mo_energy))):
            idx = e_idx[i]
            occ_str = "occ" if mo_occ[idx] > 0 else "vir"
            print(f"  MO {i+1:3d}: epsilon = {mo_energy[idx]: 12.6f}  [{occ_str}]")
        break

    dm = dm_new
    e_old = e_tot
else:
    print("WARNING: SCF did not converge!")


Iter           E_total          dE          dD
----------------------------------------------
   1    -68.8738900069   6.887e+01   1.531e+01
   2    -69.9486889568   1.075e+00   1.469e+01
   3    -73.3050793488   3.356e+00   1.790e+00
   4    -75.7504819496   2.445e+00   7.180e-01
   5    -76.0246557221   2.742e-01   7.765e-02
   6    -76.0265916349   1.936e-03   2.477e-02
   7    -76.0267567714   1.651e-04   6.305e-03
   8    -76.0267654457   8.674e-06   9.580e-04
   9    -76.0267656686   2.228e-07   1.484e-04
  10    -76.0267656729   4.382e-09   1.728e-05
  11    -76.0267656731   1.624e-10   5.757e-06
  12    -76.0267656731   8.015e-12   1.004e-06
  13    -76.0267656731   3.553e-13   1.481e-07
  14    -76.0267656731   5.684e-14   1.289e-08
  15    -76.0267656731   2.842e-14   9.703e-10

Converged in 15 iterations!
E_total (RHF+DIIS/cc-pVDZ) = -76.0267656731 Hartree
E_elec = -85.2150240909
E_nn   = 9.1882584177

Orbital energies (Hartree):
  MO   1: epsilon =   -20.550598  [occ]
  MO 

---

## 7 · 与 PySCF 对照


In [35]:
from pyscf import scf

mf_ref = scf.RHF(mol)
mf_ref.kernel()
print(f"\nPySCF RHF:     {mf_ref.e_tot:.10f} Hartree")
print(f"Our RHF+DIIS:   {e_tot:.10f} Hartree")
print(f"Difference:     {abs(mf_ref.e_tot - e_tot):.2e} Hartree")


converged SCF energy = -76.0267656731175

PySCF RHF:     -76.0267656731 Hartree
Our RHF+DIIS:   -76.0267656731 Hartree
Difference:     2.49e-12 Hartree


---

## 8 · 总结

| 模块 | 核心公式 |
|------|---------|
| **Fock 矩阵** | $\mathbf{F} = \mathbf{H}^{\text{core}} + \mathbf{J} - \frac{1}{2}\mathbf{K}$ |
| **Coulomb (J)** | $J_{\mu\nu} = \sum_{\kappa\lambda} D_{\kappa\lambda}(\mu\nu\vert \kappa\lambda)$ |
| **Exchange (K)** | $K_{\mu\nu} = \sum_{\kappa\lambda} D_{\kappa\lambda}(\mu\kappa\vert \nu\lambda)$ |
| **密度矩阵** | $\mathbf{D} = \mathbf{C}_{\text{occ}} \mathbf{n} \mathbf{C}_{\text{occ}}^T$ |
| **Roothaan 方程** | $\mathbf{F}\mathbf{C} = \mathbf{S}\mathbf{C}\boldsymbol{\varepsilon}$ |
| **DIIS 误差** | $\mathbf{e} = \mathbf{S}^{-1/2}(\mathbf{FDS} - \mathbf{SDF})\mathbf{S}^{-1/2}$ |
| **DIIS 外推** | $\mathbf{F}^{\text{DIIS}} = \sum_i c_i \mathbf{F}_i$ |

后续的 UHF、MP2、CCSD 都将以此 RHF 为基础。
